# Patient-Level Benchmarks

This notebook reproduces the patient-level numbers presented in the release/blog from checked-in artifacts: CRC MOA-tailored rank score, cSCC checkpoint compartment score, and CRC module mean cosine readout.

It also regenerates patient-level figures under `artifacts/notebook_figures/`.

In [ ]:
from pathlib import Path
import csv
import html
import json
import math
import sys

try:
    from IPython.display import SVG, Markdown, display
except Exception:
    SVG = Markdown = None
    def display(value):
        print(value)

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Run this notebook from inside the open-benchmarks repo")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from spatial_benchmarks.metrics import finite_float
from spatial_benchmarks.reproduce import (
    reproduce_crc_module_mean_cosine,
    reproduce_crc_moa_tailored_rank_score,
    reproduce_cscc_checkpoint_compartment,
    raise_if_drift,
)

FIG_DIR = ROOT / "artifacts/notebook_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def read_rows(path):
    with Path(path).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

def show_svg(path):
    if SVG is None:
        print(path)
    else:
        display(SVG(filename=str(path)))

def write_svg(path, body, width=760, height=520):
    svg = f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">{body}</svg>'
    path.write_text(svg + "\n", encoding="utf-8")
    return path

## Recompute Metrics

In [ ]:
CRC = ROOT / "patient-level-bench/model_scores/crc_moa_tailored_20260525"
CSCC = ROOT / "patient-level-bench/model_scores/cscc_checkpoint_compartment_20260604"
COSINE = ROOT / "patient-level-bench/observed_readouts/crc_module_mean_cosine_20260604"

crc_report = reproduce_crc_moa_tailored_rank_score(
    scores_csv=CRC / "crc_patient_moa_tailored_rank_scores_20260525.csv",
    metrics_csv=CRC / "crc_patient_moa_tailored_metrics_20260525.csv",
)
cscc_report = reproduce_cscc_checkpoint_compartment(
    scores_csv=CSCC / "cscc_checkpoint_compartment_patient_scores_20260604.csv",
    metrics_csv=CSCC / "cscc_checkpoint_compartment_metrics_20260604.csv",
)
cosine_report = reproduce_crc_module_mean_cosine(
    module_vectors_csv=COSINE / "crc_module_mean_cosine_module_vectors.csv",
    patient_steps_csv=COSINE / "crc_module_mean_cosine_patient_steps.csv",
    step_summary_csv=COSINE / "crc_module_mean_cosine_step_summary.csv",
)

for report in (crc_report, cscc_report, cosine_report):
    raise_if_drift(report)

patient_headlines = {
    "CRC rows": crc_report["metrics"]["n"],
    "CRC AUC response high": round(crc_report["metrics"]["auc_response_high"], 3),
    "CRC fixed 0.5 balanced accuracy": round(crc_report["metrics"]["fixed_0p5_balanced_accuracy"], 3),
    "cSCC rows": cscc_report["metrics"]["n_patients"],
    "cSCC AUC response high": round(cscc_report["metrics"]["auc_response_high"], 3),
    "cSCC Spearman response high": round(cscc_report["metrics"]["spearman_response_high"], 3),
    "CRC cosine best step": cosine_report["best_step"]["cascade_step"],
    "CRC cosine step 1 mean": round(cosine_report["step_summary"][0]["mean"], 3),
    "CRC cosine step 4 mean": round(cosine_report["best_step"]["mean"], 3),
}
patient_headlines

## Figure Helpers

In [ ]:
def patient_bar_chart(rows, score_col, label_col, title, output_name, y_max=None):
    width, height = 840, 500
    left, right, top, bottom = 72, 26, 58, 96
    plot_w = width - left - right
    plot_h = height - top - bottom
    values = [finite_float(row.get(score_col)) for row in rows]
    y_max = y_max or max(values) * 1.15 or 1.0
    bar_w = plot_w / max(1, len(rows)) * 0.72
    def sy(value):
        return top + plot_h * (1 - value / y_max)
    body = [
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{left}" y="32" font-family="Inter, Arial" font-size="22" font-weight="700" fill="#202124">{html.escape(title)}</text>',
        f'<line x1="{left}" y1="{top + plot_h}" x2="{left + plot_w}" y2="{top + plot_h}" stroke="#333"/>',
        f'<line x1="{left}" y1="{top}" x2="{left}" y2="{top + plot_h}" stroke="#333"/>',
    ]
    for tick_i in range(6):
        tick = y_max * tick_i / 5
        y = sy(tick)
        body.append(f'<line x1="{left}" y1="{y:.1f}" x2="{left + plot_w}" y2="{y:.1f}" stroke="#eef0f2"/>')
        body.append(f'<text x="{left - 10}" y="{y + 4:.1f}" text-anchor="end" font-family="Inter, Arial" font-size="12" fill="#5f6368">{tick:.2f}</text>')
    if y_max >= 0.5:
        y = sy(0.5)
        body.append(f'<line x1="{left}" y1="{y:.1f}" x2="{left + plot_w}" y2="{y:.1f}" stroke="#9aa0a6" stroke-dasharray="5 5"/>')
    for i, row in enumerate(rows):
        value = finite_float(row.get(score_col))
        is_response = str(row.get(label_col)).strip().lower() in {"1", "true", "responder", "partial response", "pcr", "mpr"}
        color = "#2f6f9f" if is_response else "#b84a3a"
        x = left + plot_w * (i + 0.5) / len(rows) - bar_w / 2
        y = sy(value)
        body.append(f'<rect x="{x:.1f}" y="{y:.1f}" width="{bar_w:.1f}" height="{top + plot_h - y:.1f}" fill="{color}"/>')
        label = row.get("source_patient_id") or row.get("patient_id") or str(i + 1)
        body.append(f'<text x="{x + bar_w / 2:.1f}" y="{height - 58}" text-anchor="middle" font-family="Inter, Arial" font-size="11" fill="#333" transform="rotate(-45 {x + bar_w / 2:.1f} {height - 58})">{html.escape(str(label))}</text>')
    body.append(f'<rect x="{left}" y="{height - 20}" width="12" height="12" fill="#2f6f9f"/><text x="{left + 18}" y="{height - 10}" font-family="Inter, Arial" font-size="12" fill="#333">Responder</text>')
    body.append(f'<rect x="{left + 116}" y="{height - 20}" width="12" height="12" fill="#b84a3a"/><text x="{left + 134}" y="{height - 10}" font-family="Inter, Arial" font-size="12" fill="#333">Non-responder</text>')
    return write_svg(FIG_DIR / output_name, "".join(body), width, height)

def cosine_trajectory_chart(step_rows, output_name):
    width, height = 820, 500
    left, right, top, bottom = 74, 34, 58, 70
    plot_w = width - left - right
    plot_h = height - top - bottom
    y_min, y_max = -0.25, 0.60
    def sx(step):
        return left + plot_w * (step - 1) / 7
    def sy(value):
        return top + plot_h * (1 - (value - y_min) / (y_max - y_min))
    body = [
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{left}" y="32" font-family="Inter, Arial" font-size="22" font-weight="700" fill="#202124">CRC Module Mean Cosine by Cascade Step</text>',
        f'<line x1="{left}" y1="{sy(0):.1f}" x2="{left + plot_w}" y2="{sy(0):.1f}" stroke="#333"/>',
        f'<line x1="{left}" y1="{top}" x2="{left}" y2="{top + plot_h}" stroke="#333"/>',
    ]
    for tick in [-0.2, 0.0, 0.2, 0.4, 0.6]:
        y = sy(tick)
        body.append(f'<line x1="{left}" y1="{y:.1f}" x2="{left + plot_w}" y2="{y:.1f}" stroke="#eef0f2"/>')
        body.append(f'<text x="{left - 10}" y="{y + 4:.1f}" text-anchor="end" font-family="Inter, Arial" font-size="12" fill="#5f6368">{tick:.1f}</text>')
    series = [("mean", "All", "#202124"), ("pr_mean", "PR", "#2f6f9f"), ("sd_mean", "SD", "#b84a3a")]
    for key, label, color in series:
        points = []
        for row in step_rows:
            step = int(row["cascade_step"])
            value = finite_float(row[key])
            points.append(f'{sx(step):.1f},{sy(value):.1f}')
        point_text = " ".join(points)
        body.append(f'<polyline points="{point_text}" fill="none" stroke="{color}" stroke-width="3"/>')
        for row in step_rows:
            step = int(row["cascade_step"])
            value = finite_float(row[key])
            body.append(f'<circle cx="{sx(step):.1f}" cy="{sy(value):.1f}" r="4" fill="{color}"/>')
    for step in range(1, 9):
        body.append(f'<text x="{sx(step):.1f}" y="{height - 36}" text-anchor="middle" font-family="Inter, Arial" font-size="12" fill="#333">{step}</text>')
    for i, (_, label, color) in enumerate(series):
        x = left + i * 84
        body.append(f'<line x1="{x}" y1="{height - 16}" x2="{x + 18}" y2="{height - 16}" stroke="{color}" stroke-width="3"/>')
        body.append(f'<text x="{x + 24}" y="{height - 12}" font-family="Inter, Arial" font-size="12" fill="#333">{label}</text>')
    return write_svg(FIG_DIR / output_name, "".join(body), width, height)

## Generate Patient Figures

In [ ]:
crc_rows = read_rows(CRC / "crc_patient_moa_tailored_rank_scores_20260525.csv")
cscc_rows = read_rows(CSCC / "cscc_checkpoint_compartment_patient_scores_20260604.csv")

crc_rows = sorted(crc_rows, key=lambda row: finite_float(row["response_score_rank_calibrated"]), reverse=True)
cscc_rows = sorted(cscc_rows, key=lambda row: finite_float(row["relative_response_probability"]), reverse=True)

fig1 = patient_bar_chart(
    crc_rows,
    "response_score_rank_calibrated",
    "observed_responder",
    "CRC Patient Response Rank Score",
    "patient_crc_rank_score.svg",
    y_max=1.0,
)
fig2 = patient_bar_chart(
    cscc_rows,
    "relative_response_probability",
    "response_label",
    "cSCC Checkpoint Compartment Response Score",
    "patient_cscc_checkpoint_score.svg",
)
fig3 = cosine_trajectory_chart(cosine_report["step_summary"], "patient_crc_module_mean_cosine_trajectory.svg")

for fig in (fig1, fig2, fig3):
    show_svg(fig)

[str(fig.relative_to(ROOT)) for fig in (fig1, fig2, fig3)]